In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# ===============================
# CONFIGURATION
# ===============================

SERVICE_LEVEL = 0.95
Z = norm.ppf(SERVICE_LEVEL)

# ===============================
# LOAD FILE
# ===============================

df = pd.read_excel("indent_vs_actual_wide_analysis.xlsx")

# ===============================
# IDENTIFY DAILY INDENT COLUMNS ONLY
# ===============================

indent_cols = [
    col for col in df.columns
    if "Indent" in col
    and "Total" not in col
    and "Deviation" not in col
]

print("Indent columns used:")
print(indent_cols)

# ===============================
# CALCULATE MEAN & STD ON INDENT
# ===============================

df["Mean_Indent"] = df[indent_cols].mean(axis=1)
df["Std_Indent"] = df[indent_cols].std(axis=1)

df["Std_Indent"] = df["Std_Indent"].fillna(0)

# ===============================
# CONFIDENCE INTERVAL
# ===============================

df["Indent_CI_Lower"] = df["Mean_Indent"] - Z * df["Std_Indent"]
df["Indent_CI_Upper"] = df["Mean_Indent"] + Z * df["Std_Indent"]

df["Indent_CI_Lower"] = df["Indent_CI_Lower"].apply(lambda x: max(0, x))

# ===============================
# SAVE
# ===============================

df.to_excel("indent_confidence_band_clean.xlsx", index=False)

print("Indent confidence band calculated correctly.")


In [ ]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

# Load indent confidence interval file
ci_df = pd.read_excel("indent_confidence_band_clean.xlsx")

# Load updated actual file
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# MERGE 12th & 13th ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH FEB AGAINST INDENT CI
# =====================================

df["12th_Inside_Indent_CI"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper"])
)

# =====================================
# VALIDATE 13TH FEB AGAINST INDENT CI
# =====================================

df["13th_Inside_Indent_CI"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper"])
)

# =====================================
# HIT RATE CALCULATION
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI"].mean() * 100

print("12th Feb Hit Rate (Indent CI):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent CI):", round(hit_rate_13, 2), "%")

# Save validation file
df.to_excel("Indent_CI_validation_12_13.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD INDENT CI FILE
# =====================================

df = pd.read_excel("indent_confidence_band_clean.xlsx")

# =====================================
# RECOMPUTE INDENT CI USING ±2σ
# =====================================

Z = 2  # ±2 sigma

df["Indent_CI_Lower_2sigma"] = df["Mean_Indent"] - Z * df["Std_Indent"]
df["Indent_CI_Upper_2sigma"] = df["Mean_Indent"] + Z * df["Std_Indent"]

df["Indent_CI_Lower_2sigma"] = df["Indent_CI_Lower_2sigma"].clip(lower=0)

# =====================================
# VALIDATE 12TH FEB
# =====================================

df["12th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH FEB
# =====================================

df["13th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# HIT RATE
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (Indent ±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent ±2σ):", round(hit_rate_13, 2), "%")

# Save
df.to_excel("Indent_CI_2sigma_validation.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

ci_df = pd.read_excel("confidence_band_analysis.xlsx")
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# RECOMPUTE CI USING ±2σ
# =====================================

ci_df["CI_Lower_2sigma"] = ci_df["Mean_Actual"] - 2 * ci_df["Std_Actual"]
ci_df["CI_Upper_2sigma"] = ci_df["Mean_Actual"] + 2 * ci_df["Std_Actual"]

ci_df["CI_Lower_2sigma"] = ci_df["CI_Lower_2sigma"].clip(lower=0)

# =====================================
# MERGE 12th & 13th ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH FEB (±2σ)
# =====================================

df["12th_Inside_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH FEB (±2σ)
# =====================================

df["13th_Inside_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

# =====================================
# HIT RATE CALCULATION
# =====================================

hit_rate_12 = df["12th_Inside_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (±2σ):", round(hit_rate_13, 2), "%")

# Save validation file
df.to_excel("CI_validation_12_13_2sigma.xlsx", index=False)


In [ ]:
import pandas as pd

# =====================================
# LOAD INDENT CI FILE
# =====================================

ci_df = pd.read_excel("indent_confidence_band_clean.xlsx")
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# COMPUTE ±2σ ON INDENT
# =====================================

ci_df["Indent_CI_Lower_2sigma"] = ci_df["Mean_Indent"] - 2 * ci_df["Std_Indent"]
ci_df["Indent_CI_Upper_2sigma"] = ci_df["Mean_Indent"] + 2 * ci_df["Std_Indent"]

ci_df["Indent_CI_Lower_2sigma"] = ci_df["Indent_CI_Lower_2sigma"].clip(lower=0)

# =====================================
# MERGE 12TH & 13TH ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH (Indent ±2σ)
# =====================================

df["12th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH (Indent ±2σ)
# =====================================

df["13th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# HIT RATE
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (Indent ±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent ±2σ):", round(hit_rate_13, 2), "%")

# Save
df.to_excel("Indent_CI_2sigma_validation.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np

# =====================================
# 1. LOAD FILES
# =====================================
ci_df = pd.read_excel("confidence_band_analysis.xlsx")
actual_df = pd.read_excel("plan_actual.xlsx")

# =====================================
# 2. RECOMPUTE CI USING ±2σ (same as your original)
# =====================================
ci_df["CI_Lower_2sigma"] = ci_df["Mean_Actual"] - 2 * ci_df["Std_Actual"]
ci_df["CI_Upper_2sigma"] = ci_df["Mean_Actual"] + 2 * ci_df["Std_Actual"]
ci_df["CI_Lower_2sigma"] = ci_df["CI_Lower_2sigma"].clip(lower=0)

# =====================================
# 3. MERGE 12th & 13th ACTUAL/PLAN DATA
# =====================================
cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]
actual_subset = actual_df[cols_to_merge]
df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# 4. LOGIC SIMULATION & COMPARISON
# =====================================
results = []

for _, row in df.iterrows():
    material = row["Material"]
    mu = row["Mean_Actual"]
    lower = row["CI_Lower_2sigma"]
    upper = row["CI_Upper_2sigma"]
    
    if mu <= 0:
        continue
    
    # Start simulation with zero inventory
    inventory = 0.0
    
    for date_str in ["2026-02-12", "2026-02-13"]:
        plan_col = f"{date_str} Total Production Plan"
        
        # Use the plan value as "demand/forecast" for comparison
        D = row[plan_col] if pd.notna(row[plan_col]) else mu
        
        # ── Buffer rule (your logic) ──
        days_cover = inventory / mu if mu > 0 else 0.0
        if days_cover >= 3.0:
            target_buffer = 0
        else:
            target_buffer = int(np.floor(days_cover)) + 1   # 0.xx→1, 1.xx→2, 2.xx→3
        
        target_inv = target_buffer * mu
        extra = max(0.0, target_inv - inventory)
        wish = D + extra
        
        # ── Apply stability clamp ──
        P = max(lower, min(upper, wish))
        P = round(P, 0)
        
        # Reason string
        reasons = []
        if target_buffer > 0:
            reasons.append(f"Build {target_buffer}d buffer")
        if P > D:
            reasons.append(f"+{int(P - D)} extra")
        elif P < D:
            reasons.append("Draw from stock")
        if P >= upper:
            reasons.append("Upper cap")
        elif P <= lower:
            reasons.append("Lower cap")
        reason = " | ".join(reasons) or "Steady at mean"
        
        # Projected inventory using logic's production
        end_inv = max(0.0, inventory + P - D)
        
        # Differences vs your actual plan
        diff_units = P - D
        diff_pct = (diff_units / D * 100) if D > 0 else 0
        
        results.append({
            "Material": material,
            "Date": date_str,
            "Your_Plan_Qty": D,
            "Logic_Qty": P,
            "Diff_Units": diff_units,
            "Diff_Percent": round(diff_pct, 2),
            "Days_Cover_Start": round(days_cover, 2),
            "Target_Buffer": target_buffer,
            "End_Inv_Logic": round(end_inv, 1),
            "Reason": reason,
            "Plan_Inside_Band": (lower <= D <= upper)
        })
        
        # Carry forward inventory using logic's decision
        inventory = end_inv

# =====================================
# 5. CREATE REPORT
# =====================================
report = pd.DataFrame(results)

# Keep original hit rates for reference
df["12th_Inside_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["CI_Upper_2sigma"])
)
df["13th_Inside_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

hit_12 = df["12th_Inside_CI_2sigma"].mean() * 100
hit_13 = df["13th_Inside_CI_2sigma"].mean() * 100

# Summary print
print("\n" + "="*70)
print("LOGIC vs YOUR PLANS – 12 & 13 Feb 2026")
print("="*70)
print(f"Materials processed                  : {len(report['Material'].unique())}")
print(f"Original hit rate 12th Feb (±2σ)     : {round(hit_12, 2)} %")
print(f"Original hit rate 13th Feb (±2σ)     : {round(hit_13, 2)} %")
print(f"Average |difference| (units)         : {round(report['Diff_Units'].abs().mean(), 1)}")
print(f"Average difference %                 : {round(report['Diff_Percent'].abs().mean(), 2)} %")
print(f"Cases where logic differs >10%       : {(report['Diff_Percent'].abs() > 10).sum()}")
print(f"Avg ending inventory 13th (logic)    : {round(report[report['Date'] == '2026-02-13']['End_Inv_Logic'].mean(), 1)}")
print("="*70)

# Save detailed report
report.to_excel("Logic_vs_Your_Plans_12_13_Detailed.xlsx", index=False)

# Pivot version - with flattened columns
pivot = report.pivot(
    index="Material",
    columns="Date",
    values=["Logic_Qty", "Your_Plan_Qty", "Diff_Percent", "Reason", "End_Inv_Logic"]
)

# Flatten MultiIndex columns
pivot.columns = ['_'.join([str(level) for level in col if level]).strip('_') 
                 for col in pivot.columns.values]

# Save pivot
pivot.reset_index().to_excel("Logic_vs_Your_Plans_12_13_Pivot.xlsx", index=False)

print("\nFiles saved:")
print("1. Logic_vs_Your_Plans_12_13_Detailed.xlsx   → row per material + date")
print("2. Logic_vs_Your_Plans_12_13_Pivot.xlsx      → one row per material")